In [ ]:
import geemap
import ee
import os
ee.Authenticate()
project = os.environ.get("GEE_PROJECT")
if project:
    ee.Initialize(project=project)
else:
    ee.Initialize()

In [ ]:
import datetime

# Definir las coordenadas del rectángulo para el recorte
#rectangle = ee.Geometry.Rectangle(
#    [-75.097, -27.654, -67.873, -36.64]  # Coordenadas: [xmin, ymin, xmax, ymax]
#)
rectangle = ee.Geometry.Rectangle([[-75.097, -27.654], [-67.873, -36.64]])

# Cargar la colección de imágenes VIIRS
viirs_collection = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG") \
            .select('avg_rad')

# Filtrar la colección por fechas y región de interés
filtered_collection = viirs_collection.filterBounds(rectangle) \
                                      .filterDate('2023-01-01', '2025-01-01')

# Convertir la colección filtrada a una lista
lista_imagenes = filtered_collection.toList(filtered_collection.size())

# Nombre de la carpeta en Google Drive
nombre_carpeta = 'viirs_avg_rad_new_years'
imagen = ee.Image(lista_imagenes.get(0))
fecha_unix = imagen.get('system:time_start').getInfo()
fecha = datetime.datetime.utcfromtimestamp(fecha_unix / 1000).strftime('%Y%m%d')
nombre_archivo = f'VIIRS_{fecha}'
task = ee.batch.Export.image.toDrive(
        image=imagen,
        description=nombre_archivo,
        folder=nombre_carpeta,
        fileNamePrefix=nombre_archivo,
        scale=463.83,  # Escala en metros
        region=rectangle,
        crs = 'EPSG:4326',
)
task.start()

# Exportar cada imagen en la lista a Google Drive
for i in range(lista_imagenes.size().getInfo()):
    imagen = ee.Image(lista_imagenes.get(i))
    fecha_unix = imagen.get('system:time_start').getInfo()
    fecha = datetime.datetime.utcfromtimestamp(fecha_unix / 1000).strftime('%Y%m%d')
    nombre_archivo = f'VIIRS{fecha}'
    task = ee.batch.Export.image.toDrive(
        image=imagen,
        description=nombre_archivo,
        folder=nombre_carpeta,
        fileNamePrefix=nombre_archivo,
        scale=463.83,  # Escala en metros
        region=rectangle,
        crs = 'EPSG:4326',
    )
    task.start()
    print(f'Tarea de exportación iniciada para {nombre_archivo}')

## Definir las coordenadas del rectángulo para el recorte
#rectangle = ee.Geometry.Rectangle(
#    [-75.077964, -28.095820, -66.554548, -26.599136]  # Coordenadas: [xmin, ymin, xmax, ymax]
#)

## Cargar la colección de imágenes VIIRS
#viirs_collection = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG") \
#            .select('avg_rad')
#collection = ee.data.getList({'id': 'NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG'})
#lista_imagenes = collection.toList(collection.size())
#nombre_carpeta = 'viirs_avg_rad'
## Filtrar la colección por fechas y región de interés
#filtered_collection = viirs_collection.filterBounds(rectangle)
#nombres_colecciones = [col['id'] for col in filtered_collection]
## Exportar la banda "stable_lights" a Google Drive
#nombre_archivo = 'VIIRS'
#print(collection)
##i=0
##for nombre in nombres_colecciones:
###    i = i+1
# Obtener la primera imagen de la colección recortada
#    image_to_download = ee.Image(filtered_collection.first())

# Obtener la URL de descarga
#    download_url = ee.batch.Export.image.toDrive(**{
#        'image':banda_stable_lights,
#        'description':nombre_archivo + name,
#        'folder':nombre_carpeta,
#        'fileNamePrefix':nombre_archivo + name,
#        'scale': 463.83,  # Escala deseada
#        'crs': 'EPSG:4326',  # Sistema de referencia de coordenadas
#        'region': rectangle,  # Región de interés
#    })
#    tarea_descarga.start()
#    print('Descargando:', nombre_archivo)
#    print('Carpeta de descarga en Google Drive:', nombre_carpeta)

In [ ]:
# Nombre de la carpeta en Google Drive
collection = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG").select('avg_rad')
lista_imagenes = collection.toList(collection.size())
nombre_carpeta = 'viirs_avg_rad_rescalado'
colecciones = ee.data.getList({'id': 'NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG'})
# Definir las coordenadas de interés
rectangle = ee.Geometry.Rectangle(
    [-75.077964, -27.621192, -67.872564, -36.569460]  # Coordenadas: [xmin, ymin, xmax, ymax]
)

                # Crear una geometría de la región de interés

# Obtener los nombres de las colecciones
nombres_colecciones = [col['id'] for col in colecciones]
# Exportar la banda "stable_lights" a Google Drive
nombre_archivo = 'VIIRS'
i=0
for nombre in nombres_colecciones:
  i = i+1
  banda_stable_lights = ee.Image(lista_imagenes.get(i))
  name = nombre.split('/')[-1]
  tarea_descarga = ee.batch.Export.image.toDrive(**{
      'image':banda_stable_lights,
      'description':nombre_archivo + name,
      'folder':nombre_carpeta,
      'fileNamePrefix':nombre_archivo + name,
      'scale': 927.67,  # Escala deseada
      'crs': 'EPSG:4326',  # Sistema de referencia de coordenadas
    #  'scale':850,  # Resolución espacial en metros
      'region':rectangle
  })
  tarea_descarga.start()

  print('Descargando:', nombre_archivo)
  print('Carpeta de descarga en Google Drive:', nombre_carpeta)

In [ ]:
Map = geemap.Map(basemap='SATELLITE')
# Visualizar
#l5_cuenca0_ndvi = ndvi_collection_LT5.filterBounds(cuenca0.geometry())
#imagen1= ee.Image(filtered_collection.toList(filtered_collection.size()).get(0))
#processed_image = ee.Image(processed_collection.first())
print("Proyeccion RS:",image_to_download.projection().getInfo())
#Map.centerObject(geometria_zona,8)
Map.addLayer(image_to_download, {}, name = "VIIRS")
Map